# Beyond Bits — End-to-End Pipeline (Fully Integrated, Fixed)

Run every cell top-to-bottom, in ONE sitting, in order. Do not reuse files from a previous run.

**Folder layout expected:** this notebook, `RTL/*.v` files, and `LTSpice/Modulator_with_demod.asc` should all be reachable via the relative paths below. Adjust `RTL_DIR` if needed.

**Requires:** Icarus Verilog (`iverilog`, `vvp`) installed and on your PATH. On Ubuntu/WSL: `sudo apt install iverilog`. On Windows, install from https://bleyer.org/icarus/ and add it to PATH, or use ModelSim/Vivado manually for the RTL steps instead (see README).

In [1]:
import subprocess, os

TEXT_INPUT = "My Life is Very Beautiful And I Want to Succeed in my Life together with my friends."
RTL_DIR = "../RTL"   # adjust if your folder layout differs

def run_verilog(sources, sim_name):
    """Compiles and runs a Verilog testbench via Icarus Verilog. Raises on failure."""
    compile_cmd = ["iverilog", "-o", sim_name] + sources
    r = subprocess.run(compile_cmd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"iverilog compile failed:\n{r.stderr}")
    r = subprocess.run(["vvp", sim_name], capture_output=True, text=True)
    print(r.stdout)
    if r.returncode != 0:
        raise RuntimeError(f"vvp simulation failed:\n{r.stderr}")


### Step 1: Text -> bits.txt

In [2]:
def step1_generate_input():
    print("\n[1/9] Generating input bits...")
    binary = ''.join(format(ord(c), '08b') for c in TEXT_INPUT)
    with open("bits.txt", "w") as f:
        f.write('\n'.join(binary) + '\n')
    print(f"      Text: '{TEXT_INPUT}'  ->  {len(binary)} bits written to bits.txt")
    return binary

step1_generate_input()



[1/9] Generating input bits...
      Text: 'My Life is Very Beautiful And I Want to Succeed in my Life together with my friends.'  ->  672 bits written to bits.txt


'010011010111100100100000010011000110100101100110011001010010000001101001011100110010000001010110011001010111001001111001001000000100001001100101011000010111010101110100011010010110011001110101011011000010000001000001011011100110010000100000010010010010000001010111011000010110111001110100001000000111010001101111001000000101001101110101011000110110001101100101011001010110010000100000011010010110111000100000011011010111100100100000010011000110100101100110011001010010000001110100011011110110011101100101011101000110100001100101011100100010000001110111011010010111010001101000001000000110110101111001001000000110011001110010011010010110010101101110011001000111001100101110'

### Step 2: RTL compression (bits.txt -> compressed.txt)
Runs `rle_compressor.v` + the race-fixed testbench automatically.

In [3]:
print("[2/9] Running RTL compressor (Icarus Verilog)...")
run_verilog([f"{RTL_DIR}/rle_compressor.v", f"{RTL_DIR}/tb_compressor_fixed.v"], "compress_sim")


[2/9] Running RTL compressor (Icarus Verilog)...
SUCCESS: Compression done. Check compressed.txt
../RTL/tb_compressor_fixed.v:79: $finish called at 6795000 (1ps)



### Step 3: Pad + Hamming(7,4) encode (compressed.txt -> encoded_clean.txt)
ECC is required per SOI'26 Pipeline Enhancement #3. Padding to a multiple of 4 is recorded so it can be trimmed off later.

In [4]:
def prepare_and_encode_ecc():
    print("\n[3/9] Padding + Hamming-encoding compressed.txt...")
    bits = [l.strip() for l in open("compressed.txt") if l.strip()]
    pad = (4 - (len(bits) % 4)) % 4
    padded = bits + ['0'] * pad
    with open("ecc_encode_input.txt", "w") as f:
        f.write('\n'.join(padded) + '\n')
    with open("ecc_pad.txt", "w") as f:
        f.write(str(pad))
    print(f"      {len(bits)} compressed bits -> padded to {len(padded)} (added {pad} pad bits)")
    return pad

prepare_and_encode_ecc()
run_verilog([f"{RTL_DIR}/hamming74_enc.v", f"{RTL_DIR}/tb_ecc_encode.v"], "ecc_encode_sim")



[3/9] Padding + Hamming-encoding compressed.txt...
      1785 compressed bits -> padded to 1788 (added 3 pad bits)
SUCCESS: Encoded 447 nibbles (1788 bits) -> 3129 codeword bits in encoded_clean.txt
../RTL/tb_ecc_encode.v:50: $finish called at 447000 (1ps)



### Step 4: Generate signal.pwl for LTSpice

In [5]:
V_HIGH = 5.0
BIT_PERIOD = 1e-6   # 1 microsecond per bit -- MUST match Step 6 sampling below
EDGE = 1e-9          # transition time -- must be << carrier period (100ns) so each
                      # bit is a clean flat step, not a slope

def generate_pwl(bits_path="encoded_clean.txt", out_path="signal.pwl"):
    print("\n[4/9] Generating PWL file for LTSpice...")
    bits = ''.join(l.strip() for l in open(bits_path) if l.strip())
    lines = []
    t = 0.0
    for bit in bits:
        v = V_HIGH if bit == '1' else 0.0
        # BUG FIX: the old code wrote ONE (t, v) point per bit. LTspice PWL
        # sources linearly interpolate between consecutive points, so with
        # only one point per bit every transition ramped over the FULL
        # 1us bit period instead of stepping -- the switch's 2.5V threshold
        # was crossed at essentially a random point mid-ramp, unrelated to
        # the actual bit boundary. Writing two points per bit (jump to the
        # value, then hold it until 1ns before the next bit) forces a real
        # rectangular step.
        lines.append(f"{t:.9e} {v:.1f}")
        lines.append(f"{t + BIT_PERIOD - EDGE:.9e} {v:.1f}")
        t += BIT_PERIOD
    lines.append(f"{t:.9e} 0.0")
    with open(out_path, "w") as f:
        f.write('\n'.join(lines))
    # BUG FIX: save the REAL transmitted bit count so Step 6 doesn't have to
    # guess it from the full 450us .tran window (which is longer than the
    # data on purpose, so it was picking up 51 bogus extra "bits" off the
    # flat tail after the real signal ends).
    with open("tx_bit_count.txt", "w") as f:
        f.write(str(len(bits)))
    print(f"      {len(bits)} bits -> {len(lines)} PWL points, {t*1e6:.1f} us total")
    print(f"      NOTE: your LTSpice .tran must simulate at least {t*1e6:.0f} us "
          f"(Modulator_with_demod.asc already uses 450us)")

generate_pwl()


[4/9] Generating PWL file for LTSpice...
      3129 bits -> 6259 PWL points, 3129.0 us total
      NOTE: your LTSpice .tran must simulate at least 3129 us (Modulator_with_demod.asc already uses 450us)


### Step 5: STOP -- run LTSpice now

1. Open `LTSpice/Modulator_with_demod.asc`
2. Confirm `signal.pwl` is in the same folder (relative path already fixed)
3. Run the simulation
4. **File -> Export data as text**, export `V(DEMOD_OUT)` (not `V(ch_out)` -- that's the raw noisy carrier, DEMOD_OUT is the actual demodulated envelope)
5. Save as `demod_output.txt` in this notebook's folder

Continue to Step 6 once you have that file.

### Step 6: Sample the analog waveform back into digital bits

In [6]:
import numpy as np

def parse_ltspice_wave_to_bits(spice_file="demod_output.txt", bit_period=BIT_PERIOD, out_path="demod_bits.txt"):
    print("\n[6/9] Reading analog waveform from LTSpice and thresholding...")
    time_axis, v_ch = np.loadtxt(spice_file, skiprows=1, unpack=True)

    # BUG FIX: use the ACTUAL transmitted bit count (saved by generate_pwl),
    # not total_duration/bit_period -- that always equaled 450 (the full
    # .tran length) regardless of how many real bits (399) were sent.
    with open("tx_bit_count.txt") as f:
        num_bits = int(f.read().strip())

    # BUG FIX: sample a WINDOW near the end of each bit and take its max,
    # instead of one instantaneous point at the bit's midpoint. A single
    # point can land on a ripple trough even during a genuine '1'. The
    # window is offset with a guard band (skip the first 20% of the bit)
    # so residual charge left over from a preceding '1' bit doesn't bleed
    # into a '0' bit's sample.
    guard_frac, end_frac = 0.5, 0.9
    window_maxes = []
    for i in range(num_bits):
        t0 = i * bit_period + guard_frac * bit_period
        t1 = i * bit_period + end_frac * bit_period
        mask = (time_axis >= t0) & (time_axis <= t1)
        window_maxes.append(v_ch[mask].max() if mask.any() else 0.0)
    window_maxes = np.array(window_maxes)

    # BUG FIX: adaptive threshold (midpoint between the observed low and
    # high clusters) instead of a hardcoded 0.4V that assumed a peak the
    # diode drop + RC never actually let the circuit reach.
    threshold = (window_maxes.max() + window_maxes.min()) / 2.0
    print(f"      Auto threshold: {threshold:.3f} V  "
          f"(observed low {window_maxes.min():.3f} V, high {window_maxes.max():.3f} V)")

    bits = ["1" if v >= threshold else "0" for v in window_maxes]
    with open(out_path, "w") as f:
        f.write('\n'.join(bits) + '\n')
    print(f"      Sampled {len(bits)} bits from the waveform -> {out_path}")
    return bits

parse_ltspice_wave_to_bits()


[6/9] Reading analog waveform from LTSpice and thresholding...
      Auto threshold: 0.246 V  (observed low 0.000 V, high 0.492 V)
      Sampled 3129 bits from the waveform -> demod_bits.txt


['0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '1',
 '1',
 '1',
 '1',
 '0',
 '0',
 '1',
 '0',
 '0',
 '1',
 '1',
 '0',
 '0',
 '0',
 '1',
 '0',
 '0',
 '1',
 '0',
 '1',
 '0',
 '1',
 '0',
 '1',
 '0',
 '1',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '1',
 '1',
 '1',
 '1',
 '0',
 '0',
 '1',
 '0',
 '0',
 '1',
 '1',
 '0',
 '0',
 '1',
 '0',
 '0',
 '0',
 '0',
 '1',
 '1',
 '1',
 '0',
 '0',
 '1',
 '1',
 '0',
 '0',
 '1',
 '1',
 '0',
 '1',
 '0',
 '0',
 '1',
 '1',
 '0',
 '0',
 '1',
 '1',
 '0',
 '0',
 '1',
 '0',
 '0',
 '1',
 '1',
 '0',
 '0',
 '0',
 '1',
 '0',
 '0',
 '1',
 '0',
 '1',
 '1',
 '1',
 '0',
 '1',
 '0',
 '0',
 '1',
 '1',
 '0',
 '0',
 '0',
 '0',
 '1',
 '1',
 '1',
 '0',
 '0',
 '1',
 '1',
 '0',
 '0',
 '1',
 '0',
 '0',
 '1',
 '1',
 '0',
 '0',
 '0',
 '1',
 '0',
 '0',
 '1',
 '0',
 '1',
 '0',
 '1',
 '0',
 '1',
 '0',
 '1',
 '0',
 '1',
 '1',
 '0',
 '1',
 '0',
 '0',
 '1',
 '0',
 '1',
 '1',
 '1',
 '1',
 '0',
 '0',
 '1',
 '0',
 '1',
 '0',
 '0',
 '0',
 '0',
 '1',
 '0',
 '0',
 '0',
 '0',
 '1'

### Step 7: Hamming decode (detects + corrects any channel-noise bit flips)

In [7]:
print("[7/9] Running RTL Hamming decoder...")
run_verilog([f"{RTL_DIR}/hamming74_dec.v", f"{RTL_DIR}/tb_ecc_decode.v"], "ecc_decode_sim")
print(open("ecc_decode_report.txt").read())


[7/9] Running RTL Hamming decoder...
SUCCESS: Decoded 447 codewords. 9 error(s) corrected. See ecc_decode_report.txt
../RTL/tb_ecc_decode.v:76: $finish called at 447000 (1ps)

ECC Decode Report -- Hamming(7,4)
Total codewords received: 447

Codeword #22: error at bit position 2 -- CORRECTED
Codeword #25: error at bit position 7 -- CORRECTED
Codeword #30: error at bit position 2 -- CORRECTED
Codeword #172: error at bit position 1 -- CORRECTED
Codeword #242: error at bit position 1 -- CORRECTED
Codeword #305: error at bit position 2 -- CORRECTED
Codeword #325: error at bit position 1 -- CORRECTED
Codeword #362: error at bit position 2 -- CORRECTED
Codeword #378: error at bit position 7 -- CORRECTED

Total errors detected and corrected: 9 / 447 codewords



### Step 8: Trim padding, then RTL decompression

In [8]:
def trim_and_prepare_decompress():
    print("\n[8/9] Trimming ECC padding before decompression...")
    pad = int(open("ecc_pad.txt").read())
    decoded = [l.strip() for l in open("decoded_bits.txt") if l.strip()]
    trimmed = decoded[:len(decoded) - pad] if pad else decoded
    with open("final_compressed_recovered.txt", "w") as f:
        f.write('\n'.join(trimmed) + '\n')
    print(f"      Trimmed {pad} pad bits -> {len(trimmed)} bits ready for decompression")

trim_and_prepare_decompress()
run_verilog([f"{RTL_DIR}/decompressor.v", f"{RTL_DIR}/tb_decompressor_final.v"], "final_decompress_sim")



[8/9] Trimming ECC padding before decompression...
      Trimmed 3 pad bits -> 1785 bits ready for decompression
SUCCESS: Decompression done. Check final_recovered_bits.txt
../RTL/tb_decompressor_final.v:91: $finish called at 13925000 (1ps)



### Step 9: Final verification

In [9]:
print("[9/9] Converting recovered bits back to text...")
bits = ''.join(l.strip() for l in open("final_recovered_bits.txt") if l.strip())
chars = []
for i in range(0, len(bits), 8):
    byte = bits[i:i+8]
    if len(byte) == 8:
        val = int(byte, 2)
        if 32 <= val <= 126:
            chars.append(chr(val))
final_text = ''.join(chars)

print(f"\n======== END-TO-END VERIFICATION ========")
print(f"Expected Input:  '{TEXT_INPUT}'")
print(f"Recovered Text:  '{final_text}'")
print(f"Pipeline Integrity Verified: {final_text == TEXT_INPUT}")
print(f"=========================================")


[9/9] Converting recovered bits back to text...

======== END-TO-END VERIFICATION ========
Expected Input:  'My Life is Very Beautiful And I Want to Succeed in my Life together with my friends.'
Recovered Text:  'My Life is Very Beautiful And I Want to Succeed in my Life together with my friends.'
Pipeline Integrity Verified: True
